# MVP Test Cases

这个 notebook 专门用于回归测试和临时加 case。核心逻辑已经迁移到 `merchant_growth_mvp.py`。

In [ ]:
import pandas as pd

from merchant_growth_mvp import (
    GAP_CONFIG,
    HEALTH_CONFIG,
    create_mock_inputs,
    get_merchant_row,
    parse_intent_payload,
    run_llm_orchestrated_pipeline,
    run_pipeline,
)

merchant_df, peer_benchmark = create_mock_inputs()
merchant_row = get_merchant_row(merchant_df, "M001")

## Interactive Run

在下面这个单元修改 `interactive_question`，然后运行下一格，就会完整跑一遍：意图识别 -> 分析模块 -> 输出结果。

In [ ]:
interactive_question = "为什么最近订单下降了？"
interactive_mock_mode = True

In [ ]:
interactive_result = run_llm_orchestrated_pipeline(
    merchant_row=merchant_row,
    question=interactive_question,
    peer_benchmark=peer_benchmark,
    health_config=HEALTH_CONFIG,
    gap_config=GAP_CONFIG,
    mock_mode=interactive_mock_mode,
)

print("=== Question ===")
print(interactive_question)
print()
print("=== Intent Result ===")
print(interactive_result["intent_result"])
print()
print("=== Structured Output ===")
interactive_result["structured_output"]


In [ ]:
print("=== Controlled LLM Output ===")
print(interactive_result["analysis_result"]["text"])
print()
print("=== Token Usage ===")
print("Estimated input tokens:", interactive_result["analysis_result"]["estimated_input_tokens"])
print("Mode:", interactive_result["analysis_result"]["mode"])


## Pipeline Cases

In [ ]:
test_cases = [
    {
        "name": "root_cause only returns health + gap",
        "question": "为什么最近订单下降了？",
        "expected_modules": ["health", "gap"],
        "present_keys": ["health", "gaps"],
        "absent_keys": ["actions"],
    },
    {
        "name": "action_recommendation returns gap + action",
        "question": "给我一些提升订单的建议",
        "expected_modules": ["gap", "action"],
        "present_keys": ["gaps", "actions"],
        "absent_keys": ["health"],
    },
    {
        "name": "default diagnosis returns all modules",
        "question": "帮我全面诊断一下这个商家",
        "expected_modules": ["health", "gap", "action"],
        "present_keys": ["health", "gaps", "actions"],
        "absent_keys": [],
    },
]

rows = []
for case in test_cases:
    output = run_pipeline(
        merchant_row=merchant_row,
        question=case["question"],
        peer_benchmark=peer_benchmark,
        health_config=HEALTH_CONFIG,
        gap_config=GAP_CONFIG,
    )

    assert output["plan"]["analysis_modules"] == case["expected_modules"]
    for key in case["present_keys"]:
        assert key in output
    for key in case["absent_keys"]:
        assert key not in output

    rows.append(
        {
            "case": case["name"],
            "question": case["question"],
            "modules": ", ".join(output["plan"]["analysis_modules"]),
            "returned_keys": ", ".join(k for k in ["health", "gaps", "actions"] if k in output),
        }
    )

pd.DataFrame(rows)

## Intent Parsing Cases

In [ ]:
intent_parse_cases = [
    ('{"intent": "diagnosis", "reason": "用户要整体看经营情况"}', "diagnosis"),
    ('```json\n{"intent": "root_cause", "reason": "用户在追问下降原因"}\n```', "root_cause"),
    ('模型解释如下：{"intent": "action_recommendation", "reason": "用户想要建议"}', "action_recommendation"),
]

for raw_text, expected_intent in intent_parse_cases:
    parsed = parse_intent_payload(raw_text)
    assert parsed["intent"] == expected_intent

try:
    parse_intent_payload('{"intent": "unknown", "reason": "bad"}')
    raise AssertionError("非法 intent 应该触发异常")
except ValueError:
    pass

print("Intent JSON parsing tests passed.")

## LLM Orchestrated Smoke Test

In [ ]:
llm_result = run_llm_orchestrated_pipeline(
    merchant_row=merchant_row,
    question="给我一些提升订单的建议",
    peer_benchmark=peer_benchmark,
    health_config=HEALTH_CONFIG,
    gap_config=GAP_CONFIG,
    mock_mode=True,
)

print(llm_result["intent_result"])
print()
print(llm_result["analysis_result"]["text"])